In [1]:
import os 
from dotenv import load_dotenv
load_dotenv() 
os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]="true"
os.environ["LANGCHAIN_PROJECT"]=os.getenv("LANGCHAIN_PROJECT")

In [14]:
## Data ingestion --from the website we need to scrape the data
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from pprint import pprint
from langchain_classic.chains import create_retrieval_chain

In [4]:
llm=ChatOpenAI(model="gpt-4o",base_url=os.getenv("BASE_URL"))

In [5]:
loader=WebBaseLoader("https://docs.smith.langchain.com/tutorials/Administrators/manage_spend")
docs=loader.load()

In [6]:
splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
chunks=splitter.split_documents(docs)

In [7]:
embeddings=OpenAIEmbeddings(model="text-embedding-3-large",base_url="https://aicredits.in/v1")


In [8]:
vector_store_db=FAISS.from_documents(chunks,embeddings)

In [9]:
query="How to manage spend in langchain?"
docs=vector_store_db.similarity_search(query)
pprint(docs)

[Document(id='0cf4f623-393a-4f63-8f46-3e1826949865', metadata={'source': 'https://docs.smith.langchain.com/tutorials/Administrators/manage_spend', 'title': 'LangSmith Observability - Docs by LangChain', 'description': 'Instrument your LLM application, investigate traces, and monitor performance in production with LangSmith.', 'language': 'en'}, page_content='cause, and resolve them with LangSmith Engine.For terminology and core concepts, refer to Observability concepts. For trace pricing, retention, and limits, see Usage and billing.To set up a LangSmith instance, visit the Platform setup section to choose between cloud, hybrid, or self-hosted. All options include observability, evaluation, prompt engineering, and deployment.'),
 Document(id='2dbfd6aa-dcfc-4212-b989-89716d8d8750', metadata={'source': 'https://docs.smith.langchain.com/tutorials/Administrators/manage_spend', 'title': 'LangSmith Observability - Docs by LangChain', 'description': 'Instrument your LLM application, investiga

In [10]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain 
from langchain_core.prompts import ChatPromptTemplate 

prompt=ChatPromptTemplate.from_template(
"""
Answer the fallowing question based only on the provided context : <context>{context} </context>
"""
) 

In [11]:
document_chain=create_stuff_documents_chain(llm,prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\nAnswer the fallowing question based only on the provided context : <context>{context} </context>\n'), additional_kwargs={})])
| ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.11', 'langchain-openai': '1.3.5'}}, output_version=None, profile={'name': 'GPT-4o', 'release_date': '2024-05-13', 'last_updated': '2024-08-06', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audi

In [12]:
from langchain_core.documents import Document
document_chain.invoke({
  "input" : "langsmith has two usage limits : total traces and extended",
  "context" : [Document(page_content="langsmith has two usage limits : total traces and extended")]
})

'Langsmith has two usage limits: total traces and extended.'

In [19]:
# ##Input retriever 
retriever=vector_store_db.as_retriever()
retrieval_chain=create_retrieval_chain(retriever,document_chain)


In [22]:
## Get the Reponse from the LLM 
response=retrieval_chain.invoke({
  "input" : "what is langsmith?"
})

In [24]:
response['answer']

'To set up a LangSmith instance, you can visit the Platform setup section and choose between cloud, hybrid, or self-hosted options. All of these options include observability, evaluation, prompt engineering, and deployment features. For additional setup details, refer to the provided LangSmith documentation.'